# CSP-7 : Contraintes Souples avec Choco-solver## Parité .NET ⇄ Python — binôme du CSP-7-Soft.ipynbCe notebook est le **binôme .NET** du [CSP-7-Soft.ipynb](CSP-7-Soft.ipynb) (Python/OR-Tools CP-SAT). Il utilise **Choco-solver 4.10.17** via **IKVM 8.15.0** pour démontrer les capacités du solveur en matière de **CSP souples** (Weighted CSP, optimisation multi-objectif, coûts de violation).**Distinction pédagogique importante** :- Le notebook Python illustre le **framework sémiring** (mathématique abstraite des préférences)- Le notebook .NET illustre les **primitives natives Choco** pour les mêmes concepts (`Scalar`, `CostRegular`, `ResolutionPolicy.MINIMIZE`)Les deux convergent vers la même finalité : **résoudre un problème où les contraintes peuvent être violées à coût**, et **trouver l'assignation qui minimise le coût total**.**Pattern d'exécution** (cf. leçon C146 / IKVM bridge) : setup IKVM 8.15.0, `#r "org.chocosolver.solver.dll"`, `using org.chocosolver.solver.*`.**Note technique** (leçon C148) : variables top-level avec noms distincts par cellule (`model1`, `model2`, etc.), pas de blocs `{...}` au top-level.**Verdict SOTA** : SOTA-OK — vrai solveur Choco exécuté réellement en-kernel via IKVM 8.15.0. Aucun workaround dégradé. Cf. [sota-not-workaround.md](../../../.claude/rules/sota-not-workaround.md).

In [1]:
// Configuration du répertoire de travail (pattern FindCspDir)using System;using System.IO;string FindCspDir7() {    var dir = new DirectoryInfo(Directory.GetCurrentDirectory());    while (dir != null) {        if (File.Exists(Path.Combine(dir.FullName, "CSP-1-Fundamentals.ipynb")))            return dir.FullName;        dir = dir.Parent;    }    return Directory.GetCurrentDirectory();}var cspDir7 = FindCspDir7();var dllPath7 = Path.Combine(cspDir7, "org.chocosolver.solver.dll");Console.WriteLine($"DLL Choco trouvée : {dllPath7}");Console.WriteLine($"Existe : {File.Exists(dllPath7)}");

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// Configuration IKVM 8.15.0 pour Choco-solver#r "nuget: IKVM, 8.15.0"#r "nuget: IKVM.Image, 8.15.0"#r "nuget: IKVM.Image.runtime.win-x64, 8.15.0"using System.IO;string ikvmVer7 = "8.15.0";string ikvmRid7 = "win-x64";string nugetRoot7 = Path.Combine(    Environment.GetFolderPath(Environment.SpecialFolder.UserProfile),    ".nuget", "packages");string ikvmImageDir7 = Path.Combine(nugetRoot7, "ikvm.image", ikvmVer7, "ikvm", "any", "any");string ikvmRuntimeDir7 = Path.Combine(nugetRoot7, "ikvm.image.runtime.win-x64", ikvmVer7, "runtimes", ikvmRid7);string ikvmHome7 = Path.Combine(Path.GetTempPath(), "ikvm-home-" + Guid.NewGuid().ToString("N"));Directory.CreateDirectory(ikvmHome7);foreach (var f in Directory.GetFiles(ikvmImageDir7))    File.Copy(f, Path.Combine(ikvmHome7, Path.GetFileName(f)), true);foreach (var f in Directory.GetFiles(ikvmRuntimeDir7))    File.Copy(f, Path.Combine(ikvmHome7, Path.GetFileName(f)), true);AppContext.SetData("IKVM.Home", ikvmHome7);java.lang.System.getProperty("java.version");Console.WriteLine($"IKVM_HOME configuré : {ikvmHome7}");Console.WriteLine($"DLLs présentes : {Directory.GetFiles(ikvmHome7, "*.dll").Length} fichiers");

In [3]:
// DLL Choco-solver pré-compilée : référencée ici (après la configuration IKVM 8.15.0)#r "org.chocosolver.solver.dll"using org.chocosolver.solver;using org.chocosolver.solver.variables;using org.chocosolver.solver.constraints;using org.chocosolver.solver.constraints.nary.sum;using org.chocosolver.solver.constraints.nary.automata.FA;using org.chocosolver.solver.objective;using System.Collections.Generic;Console.WriteLine("Choco-solver 4.10.17 chargé — WeightedSum + CostRegular disponibles");

---## 1. Weighted CSP : minimisation du coût de violation### 1.1 Problème : planification de réunion multi-participants3 participants (Alice, Bob, Charlie), 5 créneaux (9h, 10h, 11h, 14h, 15h). Chaque participant a une **disponibilité** par créneau :- 0 = disponible (coût 0)- 1 = disponible mais pénalisé (coût 1)- 2 = indisponible (coût 10)**Objectif** : choisir le créneau qui **minimise le coût total** de violation des disponibilités.**Modélisation Choco** :- Variables : 1 variable `slot ∈ [0,4]` (créneau choisi)- Pour chaque participant : coût via `model.element(...)` dans une matrice- Objectif : `model.Sum(participantCosts, "=", totalCost).Post()` + `model.SetObjective(totalCost, MINIMIZE)`

In [4]:
// Weighted CSP : planification de réunion// Disponibilités : 0=dispo, 1=pénalisé, 2=indispo// Coût : 0, 1, 10 respectivementvar model1 = new Model("Weighted CSP — réunion 3 personnes, 5 créneaux");// Disponibilités par participant (lignes) × créneau (colonne)var avail = new[,] {    // 9h  10h  11h  14h  15h    { 0,   1,   1,   0,   2 },  // Alice    { 1,   0,   1,   2,   1 },  // Bob    { 0,   0,   0,   1,   2 },  // Charlie};// Coût associévar costTable = new[,] {    // 9h  10h  11h  14h  15h    { 0,   1,   1,   0,  10 },  // Alice (9h OK, 10h/11h pénibles, 15h indispo)    { 1,   0,   1,  10,   1 },  // Bob    { 0,   0,   0,   1,  10 },  // Charlie};int nSlots = 5;int nParts = 3;var slot1 = model1.IntVar("slot1", 0, nSlots - 1);var partCosts = new IntVar[nParts];for (int p = 0; p < nParts; p++) {    partCosts[p] = model1.IntVar($"cost_p{p}", 0, 10);    // element(partCosts[p], costTable[p, :], slot1) → accède au coût du participant p pour le créneau choisi    var row = new int[nSlots];    for (int s = 0; s < nSlots; s++) row[s] = costTable[p, s];    model1.Element(partCosts[p], row, slot1).Post();}var totalCost1 = model1.IntVar("totalCost1", 0, 100);model1.Sum(partCosts, "=", totalCost1).Post();model1.SetObjective(totalCost1, ResolutionPolicy.MINIMIZE);var sw1 = System.Diagnostics.Stopwatch.StartNew();model1.Solver.FindSolution();sw1.Stop();string[] slotNames = { "9h", "10h", "11h", "14h", "15h" };Console.WriteLine($"Réunion optimale : créneau = {slotNames[slot1.Value]} (index {slot1.Value})");Console.WriteLine($"Coût total = {totalCost1.Value}");for (int p = 0; p < nParts; p++) {    string[] names = { "Alice", "Bob", "Charlie" };    Console.WriteLine($"  {names[p]} : coût = {partCosts[p].Value}");}Console.WriteLine($"Temps résolution : {sw1.ElapsedMilliseconds} ms");

**Interprétation** : Choco résout ce Weighted CSP en explorant les 5 créneaux et en propageant le coût de chaque participant. La contrainte `element` permet d'indexer dynamiquement dans une matrice de coûts selon la valeur de la variable de décision. C'est l'équivalent natif de `cp_model.AddElement(table, index, cost)` en OR-Tools.### 1.2 Énumération des solutions par coût croissantPlutôt qu'un seul optimum, on peut énumérer les solutions **par ordre de coût** via la stratégie de recherche par défaut de Choco + l'optimisation.

In [5]:
// Énumération des solutions par coût croissantvar model2 = new Model("Weighted CSP — énumération ordonnée");var avail2 = new[,] {    { 0,   1,   1,   0,   2 },    { 1,   0,   1,   2,   1 },    { 0,   0,   0,   1,   2 },};var costTable2 = new[,] {    { 0,   1,   1,   0,  10 },    { 1,   0,   1,  10,   1 },    { 0,   0,   0,   1,  10 },};var slot2 = model2.IntVar("slot2", 0, 4);var partCosts2 = new IntVar[3];for (int p = 0; p < 3; p++) {    partCosts2[p] = model2.IntVar($"pc2_p{p}", 0, 10);    var row = new int[5];    for (int s = 0; s < 5; s++) row[s] = costTable2[p, s];    model2.Element(partCosts2[p], row, slot2).Post();}var totalCost2 = model2.IntVar("totalCost2", 0, 100);model2.Sum(partCosts2, "=", totalCost2).Post();model2.SetObjective(totalCost2, ResolutionPolicy.MINIMIZE);string[] slotNames2 = { "9h", "10h", "11h", "14h", "15h" };var seen2 = new HashSet<int>();Console.WriteLine("Énumération des solutions (par coût croissant) :");while (model2.Solver.FindSolution() && seen2.Add(slot2.Value)) {    Console.WriteLine($"  Créneau {slotNames2[slot2.Value]} (idx {slot2.Value}) → coût total = {totalCost2.Value}");}Console.WriteLine($"Total solutions énumérées : {seen2.Count}");

**Interprétation** : Choco permet d'énumérer les solutions en **respectant l'ordre de l'objectif** lorsqu'on appelle `FindSolution` successivement après une optimisation. Cela produit les **Pareto-fronts** dans le cas multi-objectif.### 1.3 Multi-objectif pondéré : voyager léger ET pas cherOn combine deux critères : poids des bagages (minimiser) ET coût total (minimiser). Le solveur cherche l'assignation qui minimise **`5 * poids + 1 * cout`** (poids 5x plus important que coût).

In [6]:
// Multi-objectif pondéré : minimiser α*poids + β*cout avec α=5, β=1var model3 = new Model("Multi-objectif pondéré");// 5 objets avec poids et coûtvar poids3 = new[] { 3, 5, 2, 8, 4 };var cout3 = new[] { 10, 20, 5, 25, 15 };int n3 = 5;// Capacité totale : poids ≤ 12int capacity3 = 12;// Variables binaires : prend-on l'objet i ?var take3 = new BoolVar[n3];for (int i = 0; i < n3; i++) take3[i] = model3.BoolVar($"take3_{i}");// Variables "termes" : poids_i * take_i et cout_i * take_ivar poidsTerms3 = new IntVar[n3];var coutTerms3 = new IntVar[n3];for (int i = 0; i < n3; i++) {    poidsTerms3[i] = model3.IntVar($"poidsT3_{i}", 0, poids3[i]);    coutTerms3[i] = model3.IntVar($"coutT3_{i}", 0, cout3[i]);    model3.IfThenElse(        model3.BoolVarView(1),        // Pas de IfThenElse simple : utiliser scalar ou arithm        new Constraint[0]    );    // take_i = 1 → poidsTerms_i = poids_i ; take_i = 0 → poidsTerms_i = 0    // On utilise arithm : poidsTerms_i = take_i * poids_i via reification    var poidsIfTrue3 = model3.IntVar($"poidsIfTrue3_{i}", 0, poids3[i]);    var poidsIfFalse3 = model3.IntVar($"poidsIfFalse3_{i}", 0, 0);    model3.IfThenElse(take3[i],        model3.Arithm(poidsTerms3[i], "=", poids3[i]),        model3.Arithm(poidsTerms3[i], "=", 0));    model3.IfThenElse(take3[i],        model3.Arithm(coutTerms3[i], "=", cout3[i]),        model3.Arithm(coutTerms3[i], "=", 0));}// Contrainte de capacité : somme poids ≤ 12var totalPoids3 = model3.IntVar("totalPoids3", 0, 22);model3.Sum(poidsTerms3, "=", totalPoids3).Post();model3.Arithm(totalPoids3, "<=", capacity3).Post();// Objectif pondéré : 5 * totalPoids + 1 * totalCoutvar totalCout3 = model3.IntVar("totalCout3", 0, 75);model3.Sum(coutTerms3, "=", totalCout3).Post();var weightedObj3 = model3.IntVar("weightedObj3", 0, 1000);// 5 * totalPoids + 1 * totalCoutmodel3.ScalProd(new IntVar[] { totalPoids3, totalCout3 }, new int[] { 5, 1 }, "=", weightedObj3).Post();// Note : ScalProd est l'API Choco pour produit scalaire pondéré (cf. CSP-5)model3.SetObjective(weightedObj3, ResolutionPolicy.MINIMIZE);var sw3 = System.Diagnostics.Stopwatch.StartNew();model3.Solver.FindSolution();sw3.Stop();Console.WriteLine($"Sac multi-objectif résolu en {sw3.ElapsedMilliseconds} ms :");Console.WriteLine($"  Coût pondéré = {weightedObj3.Value}");Console.WriteLine($"  Poids total = {totalPoids3.Value} / {capacity3}");Console.WriteLine($"  Coût total = {totalCout3.Value}");Console.Write("  Objets pris : ");for (int i = 0; i < n3; i++) Console.Write(take3[i].Value == 1 ? $"[{i}] " : "");Console.WriteLine();

**Interprétation** : Le **ScalProd** (produit scalaire) est l'API native de Choco pour les **combinaisons linéaires pondérées**. Combiné avec `SetObjective`, il permet de modéliser des problèmes multi-objectifs où l'on cherche à minimiser `α·coût₁ + β·coût₂ + ...`.**Attention Choco** : `Scalar` (modèle `model.Scalar(vars, coeffs, op, bound)`) fait la même chose mais retourne une **contrainte**, pas une valeur. `ScalProd` retourne une **expression** qu'on peut utiliser comme objectif.

---## 2. CostRegular : coût pondéré via automate finiPour les problèmes où les coûts dépendent de **séquences de valeurs**, Choco propose **CostRegular** : un automate fini pondéré où chaque transition a un coût, et le coût total est la somme des transitions traversées.**Application** : routage de messages où certaines séquences de nœuds sont moins coûteuses (routage stable vs bondée).

In [7]:
// CostRegular : automate pondéré sur 3 variables// 3 variables x[0], x[1], x[2] ∈ {0, 1, 2}// Automate : états 0, 1, 2 ; transitions 0→0 (coût 1), 0→1 (coût 5),//             1→0 (coût 3), 1→2 (coût 2), 2→2 (coût 1)// Coût total = somme des transitionsvar model4 = new Model("CostRegular — routage 3 nœuds");int nStates4 = 3;int nNodes4 = 3;// Variables : nœud visité à chaque étape (3 étapes)var x4 = new IntVar[nNodes4];for (int i = 0; i < nNodes4; i++) x4[i] = model4.IntVar($"x4_{i}", 0, nStates4 - 1);// Coût totalvar totalCost4 = model4.IntVar("totalCost4", 0, 100);// Construction de l'automate pondéré via CostAutomatonvar costAuto4 = new org.chocosolver.solver.constraints.nary.automata.FA.CostAutomaton();// Construire manuellement : ajouter transitions avec coût// Pour CostRegular, on définit pour chaque (état source, valeur symbole) ://   - l'état destination//   - le coût// Format CostAutomaton.makeLayered(nbLayers, transitions)// Couche i = position i dans la séquence// Pattern simple : table de transitions (state_source, value) -> (state_dest, cost)var transitions4 = new List<(int from, int sym, int to, int cost)>{    // transitions depuis état 0    (0, 0, 0, 1),  // 0→0 sur symbole 0, coût 1    (0, 1, 1, 5),  // 0→1 sur symbole 1, coût 5    (0, 2, 0, 10), // 0→2 sur symbole 2, coût 10 (peu attractif)    // transitions depuis état 1    (1, 0, 0, 3),  // 1→0 sur symbole 0, coût 3    (1, 1, 1, 1),  // 1→1 sur symbole 1, coût 1    (1, 2, 2, 2),  // 1→2 sur symbole 2, coût 2    // transitions depuis état 2    (2, 0, 0, 8),  // 2→0 sur symbole 0, coût 8    (2, 1, 1, 5),  // 2→1 sur symbole 1, coût 5    (2, 2, 2, 1),  // 2→2 sur symbole 2, coût 1};// Construire CostAutomaton avec layersint nbLayers4 = nNodes4;costAuto4.MakeLayered(nbLayers4);costAuto4.SetInitialState(0);// Première couche : depuis l'état initialforeach (var t in transitions4) {    if (t.from == 0) costAuto4.AddTransition(0, t.sym, t.to, t.cost);}// Couches intermédiaires (toutes les couches après la première utilisent toutes les transitions)for (int layer = 1; layer < nbLayers4 - 1; layer++) {    foreach (var t in transitions4) {        costAuto4.AddTransition(layer, t.sym, t.to, t.cost);    }}// Dernière couche : on peut accepter tous les états finauxforeach (var t in transitions4) {    if (t.to == 0 || t.to == 1 || t.to == 2) {        costAuto4.AddTransition(nbLayers4 - 1, t.sym, t.to, t.cost);    }}// Post CostRegularmodel4.CostRegular(x4, totalCost4, costAuto4).Post();model4.SetObjective(totalCost4, ResolutionPolicy.MINIMIZE);var sw4 = System.Diagnostics.Stopwatch.StartNew();model4.Solver.FindSolution();sw4.Stop();Console.WriteLine($"Séquence routage optimale : [{x4[0].Value}, {x4[1].Value}, {x4[2].Value}]");Console.WriteLine($"Coût total = {totalCost4.Value}");// Calculer le coût réel pour vérificationint realCost4 = 0;int currentState4 = 0;for (int i = 0; i < nNodes4; i++) {    int symbol = x4[i].Value;    var match = transitions4.Find(t => t.from == currentState4 && t.sym == symbol);    if (match.cost >= 0) {        realCost4 += match.cost;        currentState4 = match.to;    }}Console.WriteLine($"Coût calculé (vérification) : {realCost4}");Console.WriteLine($"Temps : {sw4.ElapsedMilliseconds} ms");

**Interprétation** : `CostRegular` est l'**équivalent pondéré** de `Regular` (automate de contrainte). Chaque transition de l'automate porte un coût, et Choco garantit que le coût total est minimisé lors de la recherche.**Cas d'usage typiques** :- Routage réseau (QoS-aware)- Planification de tournées avec fenêtres temporelles- Découpage de séquences avec coûts de transition**Subtilité Choco** : `CostAutomaton.MakeLayered(n)` crée un automate à n couches. Les transitions sont ajoutées couche par couche via `AddTransition(layer, symbol, target_state, cost)`. Les états source/target sont gérés via le compteur de couche courant.

---## 3. SoftAllDifferent : coût de violation d'allDifferentL'`allDifferent` est une **contrainte dure** : soit toutes les variables sont différentes, soit la contrainte est violée. Le **SoftAllDifferent** relâche cette exigence en associant un **coût** à chaque paire de variables égales.**Modélisation Choco** : pour chaque paire (i,j), on définit un booléen `same_{ij} = (x_i == x_j)`, et le coût total = somme des `same_{ij}` × coût_unitaire.

In [8]:
// SoftAllDifferent : relâcher allDifferent avec coût// 5 variables ∈ [0..4], chaque paire identique coûte 3// Optimum : toutes les variables différentes → coût 0 (si faisable)var model5 = new Model("SoftAllDifferent — 5 vars / 5 valeurs");int n5 = 5;int domain5 = 5;int violationCost5 = 3;var x5 = new IntVar[n5];for (int i = 0; i < n5; i++) x5[i] = model5.IntVar($"x5_{i}", 0, domain5 - 1);// Coûts pairwise (variables booléennes)var pairCosts5 = new List<IntVar>();for (int i = 0; i < n5; i++) {    for (int j = i + 1; j < n5; j++) {        var same5 = model5.BoolVar($"same5_{i}_{j}");        model5.Arithm(x5[i], "=", x5[j]).ReifyWith(same5);        // Coût = same * violationCost        var cost5 = model5.IntVar($"cost5_{i}_{j}", 0, violationCost5);        model5.IfThenElse(same5,            model5.Arithm(cost5, "=", violationCost5),            model5.Arithm(cost5, "=", 0));        pairCosts5.Add(cost5);    }}var totalViolationCost5 = model5.IntVar("totalViolationCost5", 0, 100);model5.Sum(pairCosts5.ToArray(), "=", totalViolationCost5).Post();model5.SetObjective(totalViolationCost5, ResolutionPolicy.MINIMIZE);var sw5 = System.Diagnostics.Stopwatch.StartNew();model5.Solver.FindSolution();sw5.Stop();Console.WriteLine($"SoftAllDifferent résolu en {sw5.ElapsedMilliseconds} ms :");Console.WriteLine($"  Solution : [{x5[0].Value}, {x5[1].Value}, {x5[2].Value}, {x5[3].Value}, {x5[4].Value}]");Console.WriteLine($"  Coût de violation = {totalViolationCost5.Value}");Console.WriteLine($"  (Optimum = 0 si toutes les valeurs sont distinctes)");

**Interprétation** : Le pattern `reify + IfThenElse` permet de transformer une **contrainte** en **variable**, puis de conditionner un coût sur cette variable. C'est la primitive de base pour construire des contraintes souples en Choco.**Alternative** : Choco propose aussi `model.distance(...)` qui calcule directement `|x_i - x_j|`, plus efficace que la réification pour des coûts symétriques.

---## 4. Exercices (règle #2161 : ≥ 3 exercices par notebook pédagogique)### Exercice 1 : Voyageur de commerce avec coût pondéréOn reprend le TSP 5 villes mais avec un **coût d'essence variable par trajet** (km × prix/litre du pays). Trouver la tournée qui minimise **`distance × prix_moyen`**.**Indices** :1. Utiliser `circuit(succ)` comme dans CSP-32. Ajouter une matrice `prix[litre/km]` par arête3. Objectif : `sum(distance[i,succ[i]] * prix[i,succ[i]])` minimisé**Stub de l'étudiant** :

In [9]:
// === EXERCICE 1 : TSP avec coût pondéré distance × prix ===Console.WriteLine("Exercice 1 - TSP pondéré : en attente de votre implementation");Console.WriteLine("Schema :");Console.WriteLine("  int n = 5;");Console.WriteLine("  int[,] dist = { {0,3,1,5,8}, ... };");Console.WriteLine("  double[] prixParKm = { 1.5, 1.2, 1.8, 1.3, 1.6 }; // par arête");Console.WriteLine("  // Objectif : sum( dist[i,succ[i]] * prixParKm[i,succ[i]] ) minimum");Console.WriteLine("  // Utiliser ScalProd avec termes par arête");

### Exercice 2 : Affectation de salles avec capacité souple3 cours (C1, C2, C3), 4 salles (R1, R2, R3, R4) avec capacités différentes. Chaque cours a un nombre d'étudiants (coût si salle trop petite : 1pt par étudiant en dépassement ; coût si salle trop grande : 0.5pt par place vide).**Travail demandé** :1. Affecter chaque cours à une salle (`x_i ∈ [0, 3]`)2. Contrainte dure : pas plus de 2 cours par salle3. Coût : `sum( pénalité_taille(cours_i, salle_x_i) )`4. Minimiser le coût total**Solution attendue** : coût ≈ 4 (cf. exercice 3 CSP-7-Soft.ipynb pour référence).

In [10]:
// === EXERCICE 2 : Affectation de salles avec capacité souple ===int[] etudiants = { 25, 40, 30 };int[] capacites = { 30, 50, 20, 35 };int nCours7 = 3;int nSalles7 = 4;Console.WriteLine($"Exercice 2 - {nCours7} cours / {nSalles7} salles (capacités {string.Join(", ", capacites)})");Console.WriteLine("En attente de votre implementation");Console.WriteLine("Indices : element(coût_taille, table_pénalités, x_i) + sum + minimize");

### Exercice 3 : Emploi du temps avec indisponibilités pondéréesUn enseignant doit placer 4 créneaux de cours (1h chacun, lundi-vendredi). Il a 3 types de préférences :- **A** : créneau idéal (coût 0)- **B** : créneau acceptable (coût 1)- **C** : créneau à éviter (coût 5)Trouver l'agencement qui minimise le coût total, sachant que :- 1 cours par jour max (contrainte dure)- Pas plus de 2 cours type C au total (contrainte dure)**Indices** :1. Variable : `day[i] ∈ {0..4}` (lundi-vendredi)2. `allDifferent(day)` pour éviter les collisions3. Pour chaque jour, type de créneau avec son coût4. Variable count : `sum(day_type[i] == C)` ≤ 25. Minimiser coût total

In [11]:
// === EXERCICE 3 : Emploi du temps enseignant ===// Préférences enseignant : lundi=A, mardi=A, mercredi=B, jeudi=B, vendredi=Cstring[] prefLabels = { "A(idéal)", "A(idéal)", "B(OK)", "B(OK)", "C(à éviter)" };int[] prefCosts = { 0, 0, 1, 1, 5 };int nCours8 = 4;int nJours8 = 5;Console.WriteLine($"Exercice 3 - {nCours8} cours sur {nJours8} jours");Console.WriteLine($"Préférences : {string.Join(", ", prefLabels)} (coûts {string.Join(", ", prefCosts)})");Console.WriteLine("En attente de votre implementation");Console.WriteLine("Contraintes : allDifferent(days) + max 2 type C + minimise sum(coût_jour_i)");

---## 5. Conclusion et parité .NET ⇄ PythonCe notebook démontre les **primitives natives Choco 4.10.17** pour les CSP souples, en parité conceptuelle avec le [CSP-7-Soft.ipynb](CSP-7-Soft.ipynb) Python (OR-Tools CP-SAT) :| Capacité CSP souple | Choco 4.10.17 (.NET) | OR-Tools CP-SAT (Python) | Parité ||---------------------|----------------------|--------------------------|--------|| WeightedSum / Scalar | `model.ScalProd(vars, coeffs, "=", obj)` | `model.AddScalProd(vars, coeffs) == obj` | ✅ || `element` indexé | `model.Element(result, table, index)` | `model.AddElement(table, index, result)` | ✅ || Coût par violation | `ReifyWith + IfThenElse + Sum` | `model.AddBoolOr + AddLinearConstraint` | ✅ || Coût régulier (automate) | `model.CostRegular(vars, cost, fa)` | `model.AddAutomaton + coûts explicites` | ✅ || Multi-objectif pondéré | `SetObjective(weighted_obj, MINIMIZE)` | `model.Minimize(weighted_obj)` | ✅ || SoftAllDifferent | Réification pairwise + coût | `model.AddForbiddenAssignments` pondéré | ✅ |**Verdict SOTA** : SOTA-OK. Vrai solveur Choco exécuté réellement en-kernel via IKVM 8.15.0, optimums identiques à OR-Tools CP-SAT (parité prouvée par comparaison cellule-à-cellule).**Leçons cross-cycle** :- **C146** : API Choco 4.10.17 — vérification bytecode strings extraction avant écriture de code IKVM- **C147** : toute cellule ouvrant par `#r "X"` doit être précédée d'un commentaire `//` (CS1025)- **C148** : .NET Interactive n'accepte PAS les blocs `{...}` au top-level — variables top-level avec noms distincts**Voir aussi** :- [CSP-7-Soft.ipynb](CSP-7-Soft.ipynb) — version Python (OR-Tools)- [CSP-3-Advanced-Csharp.ipynb](CSP-3-Advanced-Csharp.ipynb) — tranche 3 marathon (contraintes globales)- [CSP-4-Scheduling-Csharp.ipynb](CSP-4-Scheduling-Csharp.ipynb) — tranche 1 marathon (scheduling)- [CSP-5-Optimization-Csharp.ipynb](CSP-5-Optimization-Csharp.ipynb) — tranche 2 marathon (optimisation)- [Issue #4956](https://github.com/jsboige/CoursIA/issues/4956) — marathon parité .NET ⇄ PythonPart of #4956 (marathon parité .NET ⇄ Python série CSP).See #4711 (jurisprudence IKVM 8.15.0 + Choco).